# DigiSteel-YOLO: Fully Automated Ablation Study

## Goal: Beat All 11 Reference Papers

This notebook runs **all experiments automatically** and generates a comprehensive report.

**Experiments:**
1. Baseline YOLOv11n (200 epochs)
2. C3Ghost Architecture (100 epochs)
3. Image Size 800 (100 epochs)
4. Image Size 1280 (100 epochs)
5. Enhanced Augmentation (100 epochs)
6. Cosine Learning Rate (100 epochs)
7. Final Optimized Model (200 epochs)

**Total Training Time:** ~10-12 hours on T4 GPU

**Just run all cells and wait for results!**

In [ ]:
#@title 1. Setup Environment { display-mode: "form" }

import os
import json
import time
from pathlib import Path

# Install dependencies
!pip install -q ultralytics albumentations

# Configure Kaggle
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
credentials = {"username": "hazemelerefy", "key": "KGAT_0e5696318d7e5a3caf038db9497466e5"}
(kaggle_dir / "kaggle.json").write_text(json.dumps(credentials))
os.environ['KAGGLE_USERNAME'] = credentials['username']
os.environ['KAGGLE_KEY'] = credentials['key']

# Clone repo
os.chdir('/content')
if not Path('DigiSteel-YOLO').exists():
    !git clone https://github.com/hazemelerefey/DigiSteel-YOLO.git
os.chdir('DigiSteel-YOLO')

# Download datasets if needed
if not Path('datasets/NEU-DET/yolo/images/train').exists():
    !kaggle datasets download -d sovitrath/neu-steel-surface-defect-detect-trainvalid-split -p datasets/NEU-DET --unzip -q
    !python tools/prepare_datasets.py

import torch
print(f"\n✓ Environment ready")
print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"  CUDA: {torch.version.cuda}")
print(f"  PyTorch: {torch.__version__}")

In [ ]:
#@title 2. Run Complete Ablation Study { display-mode: "form" }

import json
import time
from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import torch

# Configuration
DATASET = "NEU-DET"
SEED = 42
EPOCHS = 100
BASELINE_EPOCHS = 200

# Results storage
all_results = {}

def train_and_evaluate(name, model_config, imgsz=640, batch=16, 
                       cos_lr=False, mosaic=1.0, mixup=0.0, copy_paste=0.0,
                       epochs=EPOCHS, patience=30):
    """Train a model variant and return results."""
    
    print(f"\n{'='*60}")
    print(f"  Training: {name}")
    print(f"  Config: {model_config}")
    print(f"  Image Size: {imgsz}")
    print(f"  Batch Size: {batch}")
    print(f"  Epochs: {epochs}")
    print(f"{'='*60}")
    
    run_name = f"ablation_{name.lower().replace(' ', '_')}_{DATASET.lower()}_seed{SEED}"
    config_path = f"configs/{DATASET.lower().replace('-', '_')}.yaml"
    
    # Load model
    if model_config.endswith('.yaml'):
        model = YOLO(model_config)
    else:
        model = YOLO(model_config)
    
    # Train
    start_time = time.time()
    try:
        results = model.train(
            data=config_path,
            epochs=epochs,
            imgsz=imgsz,
            batch=batch,
            seed=SEED,
            project='runs',
            name=run_name,
            exist_ok=True,
            verbose=True,
            patience=patience,
            cos_lr=cos_lr,
            mosaic=mosaic,
            mixup=mixup,
            copy_paste=copy_paste,
        )
        training_time = time.time() - start_time
        
        # Get results
        results_csv = f"runs/{run_name}/results.csv"
        if Path(results_csv).exists():
            df = pd.read_csv(results_csv)
            best_map = df["metrics/mAP50(B)"].max()
            best_map50_95 = df["metrics/mAP50-95(B)"].max()
            
            result = {
                'experiment': name,
                'mAP50': float(best_map),
                'mAP50_95': float(best_map50_95),
                'training_time_hours': training_time / 3600,
                'imgsz': imgsz,
                'batch': batch,
                'epochs': epochs,
                'config': model_config,
            }
            
            print(f"\n  Results for {name}:")
            print(f"    mAP@0.5: {best_map:.1%}")
            print(f"    mAP@0.5:0.95: {best_map50_95:.1%}")
            print(f"    Training time: {training_time/3600:.1f} hours")
            
            return result
        else:
            print(f"  Warning: Results file not found: {results_csv}")
            return None
            
    except Exception as e:
        print(f"  Error training {name}: {e}")
        return None

# Create C3Ghost config
c3ghost_config = """
nc: 6

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 1, C3Ghost, [128]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 1, C3Ghost, [256]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 1, C3Ghost, [512]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 1, C3Ghost, [1024]]
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [512, False]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [256, False]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [512, False]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 1, C3Ghost, [1024, False]]
  - [[15, 18, 21], 1, Detect, [nc]]
"""

Path('configs').mkdir(exist_ok=True)
Path('configs/yolov11n_c3ghost.yaml').write_text(c3ghost_config)

print("="*60)
print("  STARTING AUTOMATED ABLATION STUDY")
print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"  Dataset: {DATASET}")
print(f"  Seed: {SEED}")
print("="*60)

# Experiment 1: C3Ghost Architecture
result_c3ghost = train_and_evaluate(
    name="C3Ghost_Architecture",
    model_config="configs/yolov11n_c3ghost.yaml",
    epochs=EPOCHS
)
if result_c3ghost:
    all_results['C3Ghost'] = result_c3ghost

# Experiment 2: Image Size 800
result_img800 = train_and_evaluate(
    name="Image_Size_800",
    model_config="yolo11n.pt",
    imgsz=800,
    batch=12,
    epochs=EPOCHS
)
if result_img800:
    all_results['ImgSize800'] = result_img800

# Experiment 3: Image Size 1280
result_img1280 = train_and_evaluate(
    name="Image_Size_1280",
    model_config="yolo11n.pt",
    imgsz=1280,
    batch=8,
    epochs=EPOCHS
)
if result_img1280:
    all_results['ImgSize1280'] = result_img1280

# Experiment 4: Enhanced Augmentation
result_aug = train_and_evaluate(
    name="Enhanced_Augmentation",
    model_config="yolo11n.pt",
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    epochs=EPOCHS
)
if result_aug:
    all_results['EnhancedAug'] = result_aug

# Experiment 5: Cosine Learning Rate
result_coslr = train_and_evaluate(
    name="Cosine_LR",
    model_config="yolo11n.pt",
    cos_lr=True,
    epochs=EPOCHS
)
if result_coslr:
    all_results['CosineLR'] = result_coslr

# Save all results
results_path = 'evals/ablation_study_results.json'
Path('evals').mkdir(exist_ok=True)
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\n\n{'='*60}")
print("  ABLATION STUDY COMPLETE")
print(f"{'='*60}")
print(f"\nResults saved to: {results_path}")
print(f"\nTotal experiments: {len(all_results)}")

In [ ]:
#@title 3. Generate Final Report { display-mode: "form" }

import json
from pathlib import Path
import pandas as pd

# Load results
results_path = 'evals/ablation_study_results.json'
if Path(results_path).exists():
    with open(results_path) as f:
        all_results = json.load(f)
else:
    all_results = {}

# Load baseline results
baseline_csv = "runs/baseline_neu_det_seed42/results.csv"
baseline_map = 0.779
baseline_map50_95 = 0.450

if Path(baseline_csv).exists():
    df = pd.read_csv(baseline_csv)
    baseline_map = df["metrics/mAP50(B)"].max()
    baseline_map50_95 = df["metrics/mAP50-95(B)"].max()

# Generate report
report = f"""# DigiSteel-YOLO Ablation Study - Final Report

## Executive Summary

This ablation study evaluates different architectural and hyperparameter configurations
for the DigiSteel-YOLO steel defect detection system.

## Hardware & Configuration

- **GPU**: Tesla T4 (Google Colab)
- **Dataset**: NEU-DET (6 defect classes)
- **Seed**: 42 (for reproducibility)
- **Baseline Epochs**: 200
- **Ablation Epochs**: 100

## Results Comparison

| Experiment | mAP@0.5 | mAP@0.5:0.95 | Training Time | vs Baseline | Status |
|---|---|---|---|---|---|
| **Baseline (YOLOv11n)** | {baseline_map:.1%} | {baseline_map50_95:.1%} | 1.3h | — | ✅ Reference |
| **GhostConv (Previous)** | 77.2% | 44.3% | 1.3h | -0.7% | ⚠️ Decreased |
"""

# Add all experiments
for name, data in all_results.items():
    mAP50 = data.get('mAP50', 0)
    mAP50_95 = data.get('mAP50_95', 0)
    time_h = data.get('training_time_hours', 0)
    diff = mAP50 - baseline_map
    status = "✅ Improved" if diff > 0 else "⚠️ Decreased" if diff < 0 else "➖ Same"
    
    report += f"| **{name}** | {mAP50:.1%} | {mAP50_95:.1%} | {time_h:.1f}h | {diff:+.1%} | {status} |\n"

# Find best experiment
if all_results:
    best_name = max(all_results.keys(), key=lambda k: all_results[k].get('mAP50', 0))
    best_data = all_results[best_name]
    best_map = best_data.get('mAP50', 0)
    
    report += f"""
## Key Findings

### Best Configuration: {best_name}
- **mAP@0.5**: {best_map:.1%}
- **mAP@0.5:0.95**: {best_data.get('mAP50_95', 0):.1%}
- **Training Time**: {best_data.get('training_time_hours', 0):.1f} hours
- **Improvement over Baseline**: {best_map - baseline_map:+.1%}

## Comparison with Reference Papers

| Paper | mAP@0.5 | Our Best | Difference |
|---|---|---|---|
| P10 KDM-YOLO | 95.4% | {best_map:.1%} | {best_map - 0.954:+.1%} |
| P02 LAM-YOLOv10n | 94.39% | {best_map:.1%} | {best_map - 0.9439:+.1%} |
| P03 YOLO-LSDI | 83.0% | {best_map:.1%} | {best_map - 0.83:+.1%} |
| P07 ASFRW-YOLO | 83.2% | {best_map:.1%} | {best_map - 0.832:+.1%} |
| P09 EFEN-YOLOv8 | 80.4% | {best_map:.1%} | {best_map - 0.804:+.1%} |
| P08 MSFE-YOLO | 79.8% | {best_map:.1%} | {best_map - 0.798:+.1%} |
| P04 Lightweight-YOLOv8 | 78.6% | {best_map:.1%} | {best_map - 0.786:+.1%} |
| P05 SCCI-YOLO | 78.6% | {best_map:.1%} | {best_map - 0.786:+.1%} |
| **Our Baseline** | {baseline_map:.1%} | — | — |

## Recommendations

1. **Use {best_name} configuration** for final model training
2. **Train for full 200 epochs** to maximize performance
3. **Run robustness evaluation** on the best model
4. **Export to ONNX** for edge deployment

## Next Steps

1. Train final model with best configuration for 200 epochs
2. Run robustness evaluation (6 perturbations × 4 levels)
3. Compare with all 11 reference papers
4. Export to ONNX for edge deployment
5. Generate final dissertation results

---

*Report generated automatically by DigiSteel-YOLO ablation study automation*
"""

# Save report
report_path = 'evals/ablation_study_final_report.md'
Path(report_path).write_text(report)

print(report)
print(f"\n✓ Report saved to: {report_path}")

In [ ]:
#@title 4. Train Final Optimized Model { display-mode: "form" }

from ultralytics import YOLO
import torch
import time
from pathlib import Path
import pandas as pd
import json

# Load best configuration from ablation study
results_path = 'evals/ablation_study_results.json'
if Path(results_path).exists():
    with open(results_path) as f:
        all_results = json.load(f)
    
    # Find best configuration
    best_name = max(all_results.keys(), key=lambda k: all_results[k].get('mAP50', 0))
    best_data = all_results[best_name]
    
    print(f"\n{'='*60}")
    print(f"  TRAINING FINAL OPTIMIZED MODEL")
    print(f"  Best Configuration: {best_name}")
    print(f"  Expected mAP@0.5: {best_data.get('mAP50', 0):.1%}")
    print(f"{'='*60}")
    
    # Get configuration
    config = best_data.get('config', 'yolo11n.pt')
    imgsz = best_data.get('imgsz', 640)
    batch = best_data.get('batch', 16)
    
    # Train for full 200 epochs
    run_name = f"final_optimized_{DATASET.lower()}_seed{SEED}"
    config_path = f"configs/{DATASET.lower().replace('-', '_')}.yaml"
    
    print(f"\n  Training for 200 epochs...")
    print(f"  Config: {config}")
    print(f"  Image Size: {imgsz}")
    print(f"  Batch Size: {batch}")
    
    # Load model
    if config.endswith('.yaml'):
        model = YOLO(config)
    else:
        model = YOLO(config)
    
    # Train
    start_time = time.time()
    results = model.train(
        data=config_path,
        epochs=200,
        imgsz=imgsz,
        batch=batch,
        seed=SEED,
        project='runs',
        name=run_name,
        exist_ok=True,
        verbose=True,
        patience=50,
    )
    training_time = time.time() - start_time
    
    # Get results
    results_csv = f"runs/{run_name}/results.csv"
    if Path(results_csv).exists():
        df = pd.read_csv(results_csv)
        best_map = df["metrics/mAP50(B)"].max()
        best_map50_95 = df["metrics/mAP50-95(B)"].max()
        
        print(f"\n{'='*60}")
        print(f"  FINAL MODEL RESULTS")
        print(f"{'='*60}")
        print(f"  mAP@0.5: {best_map:.1%}")
        print(f"  mAP@0.5:0.95: {best_map50_95:.1%}")
        print(f"  Training time: {training_time/3600:.1f} hours")
        print(f"\n  Best weights: runs/{run_name}/weights/best.pt")
        
        # Save final results
        final_results = {
            'model': 'DigiSteel-YOLO Final',
            'config': config,
            'mAP50': float(best_map),
            'mAP50_95': float(best_map50_95),
            'training_time_hours': training_time / 3600,
            'imgsz': imgsz,
            'batch': batch,
            'epochs': 200,
        }
        
        with open('evals/final_model_results.json', 'w') as f:
            json.dump(final_results, f, indent=2)
        
        print(f"\n  Results saved to: evals/final_model_results.json")
else:
    print("  No ablation results found. Please run the ablation study first.")